In [1]:
import sys

PROJECT_PATH = r"C:\Mestrado\Graph_Pruning\OpenGraph\link_prediction"

if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)

In [2]:
import torch as t
from torch import nn

import Utils.TimeLogger as logger
from Utils.TimeLogger import log

import numpy as np
import pickle

In [3]:
import pandas as pd

In [4]:
from model import *
from data_handler import *
from main import *

In [5]:
args.gpu = "0"

args.epoch = 0

args.load_model = "pretrn_gen1"

args.data_dir = r"C:\Mestrado\Graph_Pruning\OpenGraph\datasets"

In [6]:
def evaluate_model(exp, times=10):

    all_results = {}

    for handler in exp.multi_handler.tst_handlers:

        res_summary = {}

        for i in range(times):

            reses = exp.test_epoch(
                handler.tst_loader,
                handler
            )

            exp.add_res_to_summary(
                res_summary,
                reses
            )

            exp.multi_handler.remake_initial_projections()

        for key in res_summary:
            res_summary[key] /= times

        all_results[
            handler.data_name
        ] = res_summary

    return all_results

In [7]:
os.environ["CUDA_VISIBLE_DEVICES"] = args.gpu

if len(args.gpu.split(",")) > 1:
    args.devices = ["cuda:0", "cuda:1"]
else:
    args.devices = ["cuda:0", "cuda:0"]

args.devices = [
    t.device(device)
    for device in args.devices
]

print("Devices:", args.devices)

Devices: [device(type='cuda', index=0), device(type='cuda', index=0)]


In [8]:
trn_datasets = ["gen1"]

tst_datasets = [
    "ml1m",
    "ml10m",
    "collab"
]
if len(args.tstdata) != 0:
    tst_datasets = [args.tstdata]

if len(args.trndata) != 0:
    trn_datasets = [args.trndata]

trn_datasets = list(set(trn_datasets))
tst_datasets = list(set(tst_datasets))

print("Train datasets:", trn_datasets)
print("Test datasets:", tst_datasets)

Train datasets: ['gen1']
Test datasets: ['collab', 'ml10m', 'ml1m']


In [9]:
multi_handler = MultiDataHandler(
    trn_datasets,
    tst_datasets
)

log("Load Data")

Dataset: collab, Node num: 235868, Edge num: 1935264


C:\Mestrado\Graph_Pruning\OpenGraph\link_prediction\data_handler.py:109: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:767.)
  asym_adj = t.sparse_coo_tensor(idxs,vals,shape,check_invariants=False)


Dataset: gen1, User num: 32200, Item num: 18861, Node num: 51061, Edge num: 268007


C:\Mestrado\Graph_Pruning\OpenGraph\link_prediction\data_handler.py:65: RuntimeWarning: divide by zero encountered in power
  d_inv_sqrt = np.reshape(np.power(degree, -0.5), [-1])


Dataset: ml10m, User num: 69878, Item num: 10677, Node num: 80555, Edge num: 7200040


C:\Mestrado\Graph_Pruning\OpenGraph\link_prediction\data_handler.py:73: RuntimeWarning: divide by zero encountered in power
  d_inv_sqrt = np.reshape(np.power(col_degree, -0.5), [-1])


Dataset: ml1m, User num: 6040, Item num: 3706, Node num: 9746, Edge num: 720152
2026-09-02 19:00:22.148800: Load Data


In [10]:
exp = Exp(multi_handler)

In [11]:
exp.prepare_model()

log("Model Prepared")

Total params: 25.1904
Trainable params: 25.1904
Non-trainable params: 0.0
2026-09-02 19:02:29.309116: Model Prepared


In [12]:
if args.load_model is not None:

    exp.load_model()

Loading model from: C:\Mestrado\Graph_Pruning\OpenGraph\Models\pretrn_gen1.mod
Loading history from: C:\Mestrado\Graph_Pruning\OpenGraph\History\pretrn_gen1.his
2026-09-02 19:02:30.995482: Model Loaded


In [13]:
print(exp.model)

OpenGraph(
  (topoEncoder): TopoEncoder(
    (layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=False)
  )
  (graphTransformer): GraphTransformer(
    (gt_layers): Sequential(
      (0): GTLayer(
        (multi_head_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=1024, out_features=1024, bias=False)
        )
        (dense_layers): Sequential(
          (0): FeedForwardLayer(
            (linear): Linear(in_features=1024, out_features=1024, bias=True)
            (act): LeakyReLU(negative_slope=0.5)
          )
          (1): FeedForwardLayer(
            (linear): Linear(in_features=1024, out_features=1024, bias=True)
            (act): LeakyReLU(negative_slope=0.5)
          )
        )
        (layer_norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (layer_norm2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (fc_dropout): Dropout(p=0.1, inplace=False)
      )
      (1): GTLayer(
   

In [ ]:
#exemplo de como usar o pruner, mas não é necessário para rodar o código
#pruner = GlobalMagnitudePruner(exp.model)

#exp.model = pruner.prune(0.5)

In [14]:
baseline_results = evaluate_model(
    exp,
    times=10
)

baseline_results

{'collab': {'Recall': 0.043193541657535736,
  'NDCG': np.float64(0.020113312846467047)},
 'ml10m': {'Recall': 0.24127364657955824,
  'NDCG': np.float64(0.2652354172760908)},
 'ml1m': {'Recall': 0.18709579158681552,
  'NDCG': np.float64(0.25928402762876834)}}